In [6]:
import numpy as np
from PIL import Image
from colmap_parsing_utils import read_images_text
from numpy.typing import NDArray
from typing import List, Literal, Optional, Tuple
import math

In [7]:
colmap_images_txt="dataset/Tanks/Horse/sparse/0/images.txt"

In [8]:
images_info=read_images_text(colmap_images_txt) 
'''
images_info is a dictionart , image_id in colmap : the attribute of images that is namedtuple
("id", "qvec", "tvec", "camera_id", "name", "xys", "point3D_ids")
'''

In [ ]:

_EPS = np.finfo(float).eps * 4.0
def unit_vector(data: NDArray, axis: Optional[int] = None) -> np.ndarray:
    """Return ndarray normalized by length, i.e. Euclidean norm, along axis.

    Args:
        axis: the axis along which to normalize into unit vector
        out: where to write out the data to. If None, returns a new np ndarray
    """
    data = np.array(data, dtype=np.float64, copy=True)
    if data.ndim == 1:
        data /= math.sqrt(np.dot(data, data))
        return data
    length = np.atleast_1d(np.sum(data * data, axis))
    np.sqrt(length, length)
    if axis is not None:
        length = np.expand_dims(length, axis)
    data /= length
    return data

def quaternion_slerp(
    quat0: NDArray, quat1: NDArray, fraction: float, spin: int = 0, shortestpath: bool = True
) -> np.ndarray:
    """Return spherical linear interpolation between two quaternions.
    Args:
        quat0: first quaternion
        quat1: second quaternion
        fraction: how much to interpolate between quat0 vs quat1 (if 0, closer to quat0; if 1, closer to quat1)
        spin: how much of an additional spin to place on the interpolation
        shortestpath: whether to return the short or long path to rotation
    """
    q0 = unit_vector(quat0[:4])
    q1 = unit_vector(quat1[:4])
    if q0 is None or q1 is None:
        raise ValueError("Input quaternions invalid.")
    if fraction == 0.0:
        return q0
    if fraction == 1.0:
        return q1
    d = np.dot(q0, q1)
    if abs(abs(d) - 1.0) < _EPS:
        return q0
    if shortestpath and d < 0.0:
        # invert rotation
        d = -d
        np.negative(q1, q1)
    angle = math.acos(d) + spin * math.pi
    if abs(angle) < _EPS:
        return q0
    isin = 1.0 / math.sin(angle)
    q0 *= math.sin((1.0 - fraction) * angle) * isin
    q1 *= math.sin(fraction * angle) * isin
    q0 += q1
    return q0

In [14]:
for colmap_id,meta_data in images_info.items():
    print(colmap_id,meta_data.xys.shape,meta_data.point3D_ids.shape)

13 (3794, 2) (3794,)
12 (3815, 2) (3815,)
11 (3778, 2) (3778,)
10 (3709, 2) (3709,)
9 (3646, 2) (3646,)
8 (3703, 2) (3703,)
7 (3607, 2) (3607,)
6 (3549, 2) (3549,)
5 (3603, 2) (3603,)
4 (3653, 2) (3653,)
3 (3575, 2) (3575,)
24 (3347, 2) (3347,)
23 (3431, 2) (3431,)
22 (3648, 2) (3648,)
21 (3712, 2) (3712,)
20 (3734, 2) (3734,)
19 (3831, 2) (3831,)
18 (3897, 2) (3897,)
17 (3926, 2) (3926,)
16 (3993, 2) (3993,)
15 (3938, 2) (3938,)
2 (3613, 2) (3613,)
14 (3943, 2) (3943,)
1 (3506, 2) (3506,)
